# Constitutional AI — Harmlessness from AI Feedback (2022)

_Papers · Transformers & LLMs_

**Train a harmless assistant with almost no human harm labels: let the model critique and revise itself against written principles, then learn from its own preference judgments.**

---

This notebook is a practice scaffold for the **Constitutional AI — Harmlessness from AI Feedback (2022)** lesson. We build a tiny illustration one step at a time: a toy harm metric, the critique->revise loop run over several rounds, and then the geometric-decay shape that loop produces. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python (standard library)

This is a *conceptual* illustration of the SL-CAI critique->revise loop — our toy, not the paper's model. We build it in three steps: (1) set up a toy answer and a fake harm metric, (2) define the critique and revise operations, and (3) run the loop for several rounds and watch the harm fall with diminishing returns.

### Step 1 — A toy answer and a harm metric

To show the *shape* of the loop without a real model, we treat an "answer" as a list of word tokens and define "harm" as the count of flagged keywords still present. This stand-in lets us measure whether a revision actually reduced harm. The flagged set and the answer below are entirely made up for illustration.

In [ ]:
# Conceptual illustration of the SL-CAI critique->revise loop (OUR toy, NOT the paper).
# "Harm" here = count of flagged keywords in a toy answer.
import math
import random

random.seed(0)

# A toy "answer" as a list of word tokens; some are flagged as harmful.
FLAGGED = {"dangerous", "illegal", "toxic", "harmful", "unsafe",
           "weapon", "attack", "exploit", "poison", "harmful2"}
answer = ("here is a dangerous and illegal plan that is toxic harmful unsafe "
          "to build a weapon and attack and exploit and poison people").split()

def harm_score(tokens):
    "Fake harm metric: how many flagged keywords remain."
    return sum(t in FLAGGED for t in tokens)

print("starting harm =", harm_score(answer))

### Step 2 — Define critique and revise

The paper's loop has two moves: a **critique** that names the specific ways the answer is harmful, then a **revision** that fixes them. Our toy `critique` returns the list of flagged words, and our toy `revise` drops a fixed *fraction* (1 - `survive`) of the currently-flagged words. Removing a fraction — rather than all of them — is what produces the geometric-decay shape.

In [ ]:
def critique(tokens):
    "Fake critique: name the specific flagged words (the 'specific ways it is harmful')."
    return [t for t in tokens if t in FLAGGED]

def revise(tokens, survive=0.6):
    "Fake revision: drop a FRACTION (1 - survive) of the currently-flagged words."
    bad = [i for i, t in enumerate(tokens) if t in FLAGGED]
    n_remove = round((1 - survive) * len(bad))
    if n_remove:
        to_remove = set(random.sample(bad, n_remove))
    else:
        to_remove = set()
    return [t for i, t in enumerate(tokens) if i not in to_remove]

### Step 3 — Run the critique->revise loop

Now we iterate: each round critiques the current answer, revises it, and re-measures harm. Because every round removes the same *fraction* of an ever-smaller remainder, the harm falls monotonically but by shrinking amounts — the diminishing-returns shape the worked example walks through.

In [ ]:
print("round 0 (original): harm = %d" % harm_score(answer))

cur = answer
for k in range(1, 5):
    flags = critique(cur)            # critique step
    cur = revise(cur)                # revision step
    print("round %d: critique flagged %2d word(s) -> revised harm = %d"
          % (k, len(flags), harm_score(cur)))

# round 0 (original): harm = 10
# round 1: critique flagged 10 word(s) -> revised harm = 6
# round 2: critique flagged  6 word(s) -> revised harm = 4
# round 3: critique flagged  4 word(s) -> revised harm = 2
# round 4: critique flagged  2 word(s) -> revised harm = 1
# Harm falls every round with diminishing returns -- the geometric-decay shape of the
# worked example (survive=0.6). OUR toy illustration, NOT the paper's measured curve.

## Visualize the data & results

_If each critique->revise round removes a fixed FRACTION of the remaining harm, what shape does the 'harm remaining' trace as the number of rounds grows? Does it match the worked example's geometric decay?_

### Step 1 — Set up the geometric-decay model

The previous loop removed a fixed fraction of harm per round. We can model just that *shape* directly: start with harm = 1.0 and keep a survival fraction `r` of it each round. This isolates the geometric decay from the token bookkeeping. The value `r = 0.6` is our illustration, not a paper number.

In [ ]:
# OUR toy geometric-decay model of the critique->revise loop. r is made up.
r = 0.6                       # survival fraction of harm per round (illustration)
harm = 1.0

print("round 0: harm remaining = %.4f" % harm)
prev = harm

### Step 2 — Trace the decay and the amount removed per round

Each round multiplies the remaining harm by `r` and reports how much was removed that round. The harm decreases monotonically while the amount removed *shrinks* each time — the same qualitative shape as the paper's harmlessness-vs-revisions curve (Section 3.4), though the numbers here are ours.

In [ ]:
for k in range(1, 5):
    harm *= r
    removed = prev - harm
    print("round %d: harm remaining = %.4f   removed this round = %.4f"
          % (k, harm, removed))
    prev = harm

# round 0: harm remaining = 1.0000
# round 1: harm remaining = 0.6000   removed this round = 0.4000
# round 2: harm remaining = 0.3600   removed this round = 0.2400
# round 3: harm remaining = 0.2160   removed this round = 0.1440
# round 4: harm remaining = 0.1296   removed this round = 0.0864
# Monotone decrease with diminishing returns. OUR illustration of the SHAPE,
# NOT the paper's measured harmlessness-vs-revisions curve (Section 3.4).

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Geometric decay of harm. Model the harm remaining in an answer as starting at $1.0$, with each
            critique&rarr;revise round leaving a survival fraction $r = 0.6$. (a) What is the harm after $3$ rounds?
            (b) Does each round remove the same amount of harm? (c) What does this predict about the shape of
            the paper's harmlessness-vs-revisions curve (&sect;3.4, Figure 5)? (This survival fraction is our
            illustration, not a paper number.)

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Harm after $k$ rounds is $r^{k}$, so after $3$ rounds it is $0.6^{3}$. — _Each round multiplies the remaining harm by the same survival fraction $r$, so after $k$ rounds you have multiplied by $r$ exactly $k$ times._
- Compute $0.6^{3} = 0.216$. — _$0.6 \times 0.6 = 0.36$, and $0.36 \times 0.6 = 0.216$._
- Compare amounts removed per round: $0.40$ (round 1), $0.24$ (round 2), $0.144$ (round 3). They shrink. — _Removing a fixed fraction of an ever-smaller remainder removes a smaller absolute amount each time &mdash; diminishing returns._

**Answer:** (a) After $3$ rounds the harm is $0.6^{3} = 0.216$. (b) No &mdash; the fraction removed is
                 constant ($40\%$) but the amount shrinks each round ($0.40$, then $0.24$, then $0.144$). (c)
                 It predicts a monotonically decreasing curve with diminishing returns that approaches but
                 never reaches zero &mdash; the same qualitative shape as the paper's revisions-$1$-to-$4$
                 improvement in Figure 5 (the exact numbers there are the paper's; $r=0.6$ is ours).

</details>

**Problem 2.** RLHF vs. RLAIF (the swap). In RLHF (the InstructGPT lesson), a human looks at two answers and
            labels which is better; a reward model imitates those labels; RL optimizes the policy against the reward
            model. State precisely what Constitutional AI's RL stage (&sect;4.1) changes and what it keeps the same.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify the swapped piece: the source of the comparison label. — _RLAIF replaces the human's pick with a feedback model's choice probabilities, given the prompt, the two answers, and a randomly sampled principle (&sect;4.1)._
- Identify what is kept: train a preference model on the comparisons, then run policy-gradient RL against it. — _The preference model plays the reward model's role, and the RL optimization (the PPO family) is the same as in RLHF (&sect;4.2)._
- Note the label form: normalized, soft probabilities, not a hard 0/1. — _'we make a labeled, preference modeling comparison example with the normalized probabilities as targets' (&sect;4.1) &mdash; keeping calibration._

**Answer:** Changed: the harmlessness comparison labels come from an AI feedback model (given a random
                 constitutional principle), not from humans, and they are stored as soft normalized probabilities.
                 Kept: everything downstream &mdash; train a preference model on the comparisons, then optimize
                 the assistant against it with policy-gradient RL, exactly as in RLHF. That single substitution of
                 the label source is what the paper names "RLAIF" (&sect;4.1).

</details>

**Problem 3.** Ablation: drop the critique step. Suppose you change the SL stage so the model is asked to
            directly "rewrite this to be harmless" with no separate critique step first. Based on the paper's
            design rationale, predict what you would expect to get worse, and tie it to why the paper uses a
            critique-then-revise loop rather than a single rewrite.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that critique-then-revise is a chain-of-thought: name the harm explicitly, then fix it. — _The paper's loop has the model first identify 'specific ways in which the assistant's last response is harmful' (&sect;3.1) before rewriting, so the rewrite is targeted._
- Predict the effect of removing the critique: a less-targeted rewrite that misses subtle harms. — _Without first localizing the problem, the model has no explicit reasoning to act on, so revisions should be shallower and the resulting SL-CAI data less harmless._
- Tie to the repeatable-loop point: weaker single revisions also make extra rounds less effective. — _If each round removes less harm, the geometric decay is slower, so the revisions-1-to-4 gains (&sect;3.4) would be smaller._

**Answer:** You would expect the revised answers to be less harmless and to miss subtler harms,
                 because the critique is a chain-of-thought that localizes the specific problem before the rewrite
                 (&sect;3.1). Removing it forces an undirected "make it safer" edit, which is shallower; the
                 fine-tuned SL-CAI data is then worse, and because each round removes less harm, the benefit of
                 stacking revisions ($1$&ndash;$4$ rounds, &sect;3.4) also shrinks. The critique step is what makes
                 each revision sharp enough to compound.

</details>